In [2]:
import os
import sys
import random
import functools
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RepeatedKFold
import warnings
warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '3407'
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

def set_all_random_seeds(seed=3407):
    np.random.seed(seed)
    random.seed(seed)
    print(f"已设置随机种子为: {seed}")

def print_data_shapes(X_train, y_train, X_test, y_test):
    print(f"训练集形状: {X_train.shape}")
    print(f"测试集形状: {X_test.shape}")

def objective(trial, X_train, y_train, cv_folds, seed=3407):
    """Optuna目标函数：搜索LightGBM超参数"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 300),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
        'random_state': seed,
        'n_jobs': 1,
        'verbosity': -1,
        'boosting_type': 'gbdt'
    }
    model = lgb.LGBMRegressor(**params)
    
    rkf = RepeatedKFold(n_splits=cv_folds, n_repeats=5, random_state=seed)
    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=rkf, scoring='neg_mean_squared_error',
        n_jobs=1
    )
    mse_scores = -cv_scores
    avg_mse = np.mean(mse_scores)
    mse_std = np.std(mse_scores)
    stability_penalty = 0.1 * mse_std / avg_mse if avg_mse > 0 else 0
    return avg_mse + stability_penalty

def optimize_hyperparameters(X_train, y_train, n_trials=100, cv_folds=10, seed=3407):
    print("开始Optuna超参数优化（LightGBM）...")

    sampler = TPESampler(
        seed=seed,
        n_startup_trials=10,
        group=False,
        warn_independent_sampling=False
    )

    study = optuna.create_study(
        direction='minimize',
        study_name=f'lightgbm_regression_seed{seed}',
        storage=None,
        load_if_exists=False,
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
    )
    
    objective_partial = functools.partial(
        objective,
        X_train=X_train,
        y_train=y_train,
        cv_folds=cv_folds,
        seed=seed
    )
    
    study.optimize(objective_partial, n_trials=n_trials, n_jobs=1, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"\n最佳超参数: {best_params}")
    print(f"最佳交叉验证MSE: {study.best_value:.4f}")
    
    best_params['random_state'] = seed
    return best_params, study

def train_best_model(X_train, y_train, best_params, seed=3407):
    final_params = best_params.copy()
    final_params.update({
        'random_state': seed,
        'n_jobs': 1,
        'verbosity': -1,
        'boosting_type': 'gbdt'
    })
    model = lgb.LGBMRegressor(**final_params)
    model.fit(
        X_train, y_train,
        eval_metric='mse',
    )
    return model, (X_train, y_train)

def evaluate_model(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    mse_diff = abs(train_mse - test_mse) / train_mse * 100 if train_mse > 0 else 0
    r2_diff = abs(train_r2 - test_r2) * 100
    
    print("\n" + "="*50)
    print("模型性能评估")
    print("="*50)
    print(f"{'指标':<15} {'训练集':<15} {'测试集':<15} {'差异(%)':<10}")
    print("-"*50)
    print(f"{'MSE':<15} {train_mse:<15.4f} {test_mse:<15.4f} {mse_diff:<10.2f}")
    print(f"{'RMSE':<15} {np.sqrt(train_mse):<15.4f} {np.sqrt(test_mse):<15.4f} {'-':<10}")
    print(f"{'MAE':<15} {train_mae:<15.4f} {test_mae:<15.4f} {'-':<10}")
    print(f"{'R²':<15} {train_r2:<15.4f} {test_r2:<15.4f} {r2_diff:<10.2f}")
    print("="*50)
    
    metrics_dict = {
        'train_mse': train_mse, 'test_mse': test_mse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'train_r2': train_r2, 'test_r2': test_r2,
        'mse_diff_percent': mse_diff, 'r2_diff_percent': r2_diff
    }
    predictions = {'y_train_pred': y_train_pred, 'y_test_pred': y_test_pred}
    return metrics_dict, predictions

def save_model(model, filepath='best_lightgbm_model.pkl'):
    import joblib
    joblib.dump(model, filepath)
    print(f"模型已保存到: {filepath}")

def lightgbm_optimization_pipeline(X_train, y_train, X_test, y_test, n_trials=100, cv_folds=5, seed=3407):
    set_all_random_seeds(seed)
    print_data_shapes(X_train, y_train, X_test, y_test)
    
    best_params, study = optimize_hyperparameters(
        X_train, y_train, n_trials, cv_folds, seed
    )
    model, train_data = train_best_model(X_train, y_train, best_params, seed)
    metrics, predictions = evaluate_model(model, X_train, y_train, X_test, y_test)
    save_model(model, 'LightGBM_regression_model.pkl')
    print("\n优化完成！")
    return model, best_params, metrics, study, predictions, train_data

def main():
    SEED = 36
    
    X_train = pd.read_csv("x_train.csv")
    X_test  = pd.read_csv("x_test.csv")
    y_train = pd.read_csv("y_train.csv")
    y_test  = pd.read_csv("y_test.csv")
    
    if len(y_train.shape) > 1 and y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    if len(y_test.shape) > 1 and y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]
    
    model, best_params, metrics, study, predictions, train_data = lightgbm_optimization_pipeline(
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        n_trials=100,
        cv_folds=10,
        seed=SEED
    )
    
    print(f"\n最佳参数数量: {len(best_params)}")
    print(f"训练集R²: {metrics['train_r2']:.4f}")
    print(f"测试集R²: {metrics['test_r2']:.4f}")
    
    results = {
        'model': model, 'best_params': best_params, 'metrics': metrics,
        'study': study, 'predictions': predictions, 'train_data': train_data
    }
    return results

if __name__ == "__main__":
    results = main()

[I 2026-04-10 20:39:32,584] A new study created in memory with name: lightgbm_regression_seed36


已设置随机种子为: 36
训练集形状: (921, 30)
测试集形状: (231, 30)
开始Optuna超参数优化（LightGBM）...


Best trial: 0. Best value: 15.5428:   1%|          | 1/100 [00:14<24:12, 14.67s/it]

[I 2026-04-10 20:39:47,256] Trial 0 finished with value: 15.542805188086929 and parameters: {'n_estimators': 756, 'max_depth': 9, 'learning_rate': 0.25496836683723545, 'num_leaves': 51, 'subsample': 0.9036795649350824, 'colsample_bytree': 0.6816072982605145, 'min_child_samples': 13, 'reg_alpha': 2.1496172800756097e-06, 'reg_lambda': 0.004654371916908341, 'subsample_freq': 1}. Best is trial 0 with value: 15.542805188086929.


Best trial: 1. Best value: 12.5895:   2%|▏         | 2/100 [00:29<24:01, 14.71s/it]

...

[I 2026-04-10 20:55:57,684] Trial 97 finished with value: 11.635643750998067 and parameters: {'n_estimators': 970, 'max_depth': 7, 'learning_rate': 0.013220536282655987, 'num_leaves': 39, 'subsample': 0.6319990577173206, 'colsample_bytree': 0.5786339739441928, 'min_child_samples': 7, 'reg_alpha': 0.0008133853441106966, 'reg_lambda': 0.013965166686921512, 'subsample_freq': 5}. Best is trial 86 with value: 11.600136516536553.


Best trial: 86. Best value: 11.6001:  99%|█████████▉| 99/100 [16:45<00:16, 16.27s/it]

[I 2026-04-10 20:56:17,771] Trial 98 finished with value: 11.716574051016677 and parameters: {'n_estimators': 986, 'max_depth': 7, 'learning_rate': 0.013211782688657005, 'num_leaves': 71, 'subsample': 0.6911425502643237, 'colsample_bytree': 0.5779999681662475, 'min_child_samples': 7, 'reg_alpha': 0.00017323066182618573, 'reg_lambda': 0.03388214320422884, 'subsample_freq': 5}. Best is trial 86 with value: 11.600136516536553.


Best trial: 86. Best value: 11.6001: 100%|██████████| 100/100 [16:52<00:00, 10.13s/it]


[I 2026-04-10 20:56:25,341] Trial 99 finished with value: 14.168439796319179 and parameters: {'n_estimators': 957, 'max_depth': 7, 'learning_rate': 0.011921745664822687, 'num_leaves': 10, 'subsample': 0.6313343241675388, 'colsample_bytree': 0.6096223959802579, 'min_child_samples': 29, 'reg_alpha': 6.487112481965976e-05, 'reg_lambda': 0.007098981596341371, 'subsample_freq': 8}. Best is trial 86 with value: 11.600136516536553.

最佳超参数: {'n_estimators': 925, 'max_depth': 6, 'learning_rate': 0.012225440784966483, 'num_leaves': 58, 'subsample': 0.6788341448555861, 'colsample_bytree': 0.5836159948687595, 'min_child_samples': 5, 'reg_alpha': 5.981257620892187e-05, 'reg_lambda': 0.00022477647308359933, 'subsample_freq': 5}
最佳交叉验证MSE: 11.6001

模型性能评估
指标              训练集             测试集             差异(%)     
--------------------------------------------------
MSE             3.4534          17.4313         404.76    
RMSE            1.8583          4.1751          -         
MAE             1.242